# Partie A — Exploration, raccourcis & baseline

ArSarcasm-v2. Chargement, visualisation, nettoyage, TF-IDF, entraînement.

Le fichier de test est scellé jusqu'à la Partie F.

In [1]:
from pathlib import Path
import re
import time

import emoji
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from pyarabic.araby import DIACRITICS, TATWEEL
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_colwidth", 240)
pd.set_option("display.max_rows", 80)

SEED = 42
np.random.seed(SEED)

plt.rcParams["font.family"] = "Segoe UI"


## Chargement

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

BASE = "https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/main/ArSarcasm-v2/"
full = pd.read_csv(BASE + "training_data.csv")
test = pd.read_csv(BASE + "testing_data.csv")  # scellé jusqu'à la Partie F
full["id"], test["id"] = range(len(full)), range(len(test))
train, val = train_test_split(
    full,
    test_size=0.10,
    random_state=42,
    stratify=full["sarcasm"],
)
train = train.reset_index(drop=True).copy()
val = val.reset_index(drop=True).copy()

for frame in (full, test, train, val):
    frame["tweet"] = frame["tweet"].fillna("").astype(str)
    frame["sarcasm"] = (
        frame["sarcasm"].astype(str).str.strip().str.upper().isin(["TRUE", "1", "T", "YES"])
    )
    frame["sentiment"] = frame["sentiment"].astype(str)
    frame["dialect"] = frame["dialect"].astype(str)

# Data visualisation

### Aperçu

In [ ]:
display(full.head())
for tweet in full["tweet"].head(8):
    print("---")
    print(tweet)

### Audit du texte brut

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
MENTION_RE = re.compile(r"(?<!\w)@\w+")
HASHTAG_RE = re.compile(r"(?<!\w)#\w+")
LAUGH_RE = re.compile(r"ه{2,}")
REPEATED_PUNCT_RE = re.compile(r"([!?؟.,،؛…])\1+")
REPEATED_LETTER_RE = re.compile(r"([^\W\d_])\1{2,}")
LATIN_RE = re.compile(r"[A-Za-z]")
INDIAN_DIGIT_RE = re.compile(r"[٠-٩]")
WS_RE = re.compile(r"\s+")
DIACRITIC_CHARS = "".join(ch for ch in DIACRITICS if ch != TATWEEL)
DIACRITIC_RE = re.compile("[" + re.escape(DIACRITIC_CHARS) + "]")
TATWEEL_RE = re.compile(re.escape(TATWEEL))


def has_csv_quote_wrap(text):
    stripped = str(text).strip()
    return len(stripped) >= 2 and stripped[0] == '"' and stripped[-1] == '"'


def has_emoji(text):
    return len(emoji.emoji_lis(str(text))) > 0


CUES = [
    ("url", lambda text: URL_RE.search(text) is not None),
    ("mention", lambda text: MENTION_RE.search(text) is not None),
    ("hashtag", lambda text: HASHTAG_RE.search(text) is not None),
    ("emoji", has_emoji),
    ("laugh_heh", lambda text: LAUGH_RE.search(text) is not None),
    ("repeated_punctuation", lambda text: REPEATED_PUNCT_RE.search(text) is not None),
    ("diacritics", lambda text: DIACRITIC_RE.search(text) is not None),
    ("tatweel", lambda text: TATWEEL_RE.search(text) is not None),
    ("repeated_letters", lambda text: REPEATED_LETTER_RE.search(text) is not None),
    ("latin", lambda text: LATIN_RE.search(text) is not None),
    ("indian_digits", lambda text: INDIAN_DIGIT_RE.search(text) is not None),
    ("csv_quote_wrap", has_csv_quote_wrap),
]

tweets = full["tweet"].astype(str)
rows = []
for name, present_fn in CUES:
    present = tweets.map(present_fn).astype(bool)
    rows.append(
        {
            "feature": name,
            "n_tweets": int(present.sum()),
            "frequency": float(present.mean()),
        }
    )

audit = pd.DataFrame(rows)
display(audit)

fig, ax = plt.subplots(figsize=(8, 5))
order = audit.sort_values("frequency")
ax.barh(order["feature"], order["frequency"])
ax.set_xlabel("part des tweets")
ax.set_title("Indices de surface — fichier d'entraînement brut")
fig.tight_layout()
display(fig)
plt.close(fig)

### Sarcasme, dialecte et émotion

In [ ]:
dialect_order = ["msa", "egypt", "gulf", "levant", "magreb"]
sentiment_order = ["NEG", "NEU", "POS"]


def sarcasm_percent(frame, column, order):
    grouped = frame.assign(_y=frame["sarcasm"].astype(bool)).groupby(column, dropna=False)["_y"]
    return grouped.agg(n="size", rate="mean").reindex(order)


fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, column, order, title in (
    (axes[0], "dialect", dialect_order, "Sarcasme par dialecte"),
    (axes[1], "sentiment", sentiment_order, "Sarcasme par émotion"),
):
    stats = sarcasm_percent(full, column, order).dropna(how="all")
    ax.bar(stats.index.astype(str), stats["rate"] * 100)
    ax.set_ylim(0, 100)
    ax.set_ylabel("% sarcasme")
    ax.set_title(title)
fig.tight_layout()
display(fig)
plt.close(fig)

encoded = pd.get_dummies(full[["dialect", "sentiment"]].astype(str))
encoded.insert(0, "sarcasm", full["sarcasm"].astype(int))
order = (
    ["sarcasm"]
    + [f"dialect_{name}" for name in dialect_order]
    + [f"sentiment_{name}" for name in sentiment_order]
)
corr = encoded.corr(method="pearson").reindex(index=order, columns=order)
labels = ["sarcasme"] + [f"dialecte {name}" for name in dialect_order] + [f"émotion {name}" for name in sentiment_order]
fig, ax = plt.subplots(figsize=(8, 6.5))
image = ax.imshow(corr.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)), labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        value = corr.iloc[i, j]
        ax.text(j, i, "" if pd.isna(value) else f"{value:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Corrélation de Pearson — sarcasme, dialecte, émotion")
fig.colorbar(image, ax=ax, label="Pearson")
fig.tight_layout()
display(fig)
plt.close(fig)


### Split et distribution


In [ ]:
def sarcasm_rate_table(splits, by=None):
    frames = []
    for name, frame in splits.items():
        y = frame["sarcasm"].astype(bool)
        if by is None:
            frames.append(
                pd.DataFrame(
                    {
                        "split": [name],
                        "n": [len(frame)],
                        "n_sarcastic": [int(y.sum())],
                        "sarcasm_rate": [float(y.mean()) if len(frame) else float("nan")],
                    }
                )
            )
            continue
        grouped = frame.assign(_label=y).groupby(by, dropna=False)["_label"]
        part = grouped.agg(n="size", n_sarcastic="sum", sarcasm_rate="mean").reset_index()
        part.insert(0, "split", name)
        frames.append(part)
    return pd.concat(frames, ignore_index=True)


display(sarcasm_rate_table({"train": train, "val": val}))
display(sarcasm_rate_table({"train": train, "val": val}, by="dialect"))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_names = ["train", "val"]
plot_frames = [train, val]
counts = [int(frame["sarcasm"].sum()) for frame in plot_frames]
non_counts = [len(frame) - n for frame, n in zip(plot_frames, counts)]
x = np.arange(len(plot_names))
axes[0].bar(x - 0.18, non_counts, width=0.36, label="non sarcastique")
axes[0].bar(x + 0.18, counts, width=0.36, label="sarcastique")
axes[0].set_xticks(x, plot_names)
axes[0].set_ylabel("tweets")
axes[0].set_title("Effectifs")
axes[0].legend()
rates_plot = [n / len(frame) for n, frame in zip(counts, plot_frames)]
axes[1].bar(plot_names, rates_plot)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("taux de sarcasme")
axes[1].set_title("Taux")
fig.tight_layout()
display(fig)
plt.close(fig)


def pies(splits, column, title, order):
    fig, axes = plt.subplots(1, len(splits), figsize=(5 * len(splits), 4.2))
    for ax, (name, frame) in zip(axes, splits.items()):
        counts = frame[column].astype(str).value_counts().reindex(order).fillna(0)
        counts = counts[counts > 0]
        total = float(counts.sum())
        wedges, _texts = ax.pie(counts.to_numpy(), startangle=90, counterclock=False)
        labels = [f"{label} — {100 * float(counts[label]) / total:.1f}%" for label in counts.index]
        ax.legend(wedges, labels, loc="upper center", bbox_to_anchor=(0.5, -0.02), frameon=False)
        ax.set_title(f"{name}\nn={len(frame)}")
    fig.suptitle(title)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


pies(
    {"train": train, "val": val},
    "sentiment",
    "Sentiment (train et val)",
    ["NEG", "NEU", "POS"],
)
pies(
    {"train": train, "val": val},
    "dialect",
    "Dialecte (train et val)",
    ["msa", "egypt", "gulf", "levant", "magreb"],
)


### Vides et doublons

In [ ]:
def strip_csv_quotes(text):
    stripped = str(text).strip()
    if len(stripped) >= 2 and stripped[0] == '"' and stripped[-1] == '"':
        return stripped[1:-1].strip()
    return stripped


def normalize_for_dedup(text):
    cleaned = strip_csv_quotes(text).replace(TATWEEL, "")
    return WS_RE.sub(" ", cleaned).strip()


empty_mask = train["tweet"].map(lambda text: normalize_for_dedup(text) == "")
extra_mask = train["tweet"].duplicated(keep="first")
keys = train["tweet"].map(normalize_for_dedup)
nunique = train.assign(_key=keys).groupby("_key")["sarcasm"].nunique()
bad_keys = set(nunique[nunique > 1].index)
contradictory_mask = keys.isin(bad_keys)
test_exact = set(test["tweet"].astype(str))
test_norm = set(test["tweet"].map(normalize_for_dedup))
leak_mask = train["tweet"].astype(str).isin(test_exact) | keys.isin(test_norm)

print(f"copies en trop : {int(extra_mask.sum())}")
print(f"groupes aux deux labels : {len(bad_keys)}")
print(f"fuites test : {int(leak_mask.sum())}")

sample_keys = sorted(bad_keys)[:3]
examples = (
    train.loc[keys.isin(sample_keys), ["tweet", "sarcasm"]]
    .sort_values(["tweet", "sarcasm"])
    .drop_duplicates()
)
display(examples)


## Nettoyage

Le test officiel reste brut : on ne retire aucune de ses lignes et on ne modifie pas son texte. Les filtres portent seulement sur train et val.

1. Doublons exacts : mêmes `tweet`, `sarcasm`, `sentiment` et `dialect`. L'`id` ajouté au chargement ne fait pas partie de la comparaison. On garde la première ligne.
2. Groupes aux deux labels : la clé est le tweet normalisé (`normalize_for_dedup`, comme dans l'audit). Si cette clé porte les deux valeurs de `sarcasm` dans train+val, tout le groupe est retiré.
3. Fuites vers le test : si le tweet exact, ou le tweet normalisé, d'une ligne de train ou de val figure dans le test, on retire cette ligne. Le test n'est pas modifié.
4. Part de validation : elle doit rester voisine de 10 % du pool train+val. Si l'écart dépasse un point de pourcentage, on redécoupe seulement ce pool (90/10, stratifié sur `sarcasm`, graine 42).

Les textes vides après normalisation sont aussi retirés (le compteur vaut zéro sur ce corpus). La ponctuation est retirée plus bas, sur la colonne `text`, une fois les lignes supprimées.


In [ ]:
CONTENT_COLS = ["tweet", "sarcasm", "sentiment", "dialect"]
VAL_SHARE = 0.10
VAL_TOLERANCE = 0.01

test_before = test.copy()
ids_before = {
    "train": set(train["id"]),
    "val": set(val["id"]),
    "test": set(test["id"]),
}


def split_overview(train_df, val_df, test_df, moment):
    frames = (("train", train_df), ("val", val_df), ("test", test_df))
    total = sum(len(frame) for _, frame in frames)
    pool = len(train_df) + len(val_df)
    rows = []
    for name, frame in frames:
        labels = frame["sarcasm"].astype(bool)
        rows.append(
            {
                "moment": moment,
                "split": name,
                "n": len(frame),
                "proportion": round(len(frame) / total, 4) if total else float("nan"),
                "part_train_val": (
                    round(len(frame) / pool, 4) if name != "test" and pool else float("nan")
                ),
                "n_sarcastic": int(labels.sum()),
                "taux_sarcasme": round(float(labels.mean()), 4) if len(frame) else float("nan"),
            }
        )
    return pd.DataFrame(rows)


def _drop_row(step, split, n_before, n_dropped):
    return {
        "etape": step,
        "split": split,
        "n_avant": n_before,
        "n_retirees": n_dropped,
        "n_apres": n_before - n_dropped,
    }


overview_before = split_overview(train, val, test, "avant")
print("effectifs avant nettoyage")
display(overview_before)

steps = []
n_train, n_val = len(train), len(val)
train = train.drop_duplicates(subset=CONTENT_COLS, keep="first").reset_index(drop=True)
val = val.drop_duplicates(subset=CONTENT_COLS, keep="first").reset_index(drop=True)
steps.append(_drop_row("doublons exacts", "train", n_train, n_train - len(train)))
steps.append(_drop_row("doublons exacts", "val", n_val, n_val - len(val)))

pool = pd.concat(
    [train.assign(_split="train"), val.assign(_split="val")],
    ignore_index=True,
)
group_key = pool["tweet"].map(normalize_for_dedup)
nunique = pool.assign(_key=group_key).groupby("_key")["sarcasm"].nunique()
bad_keys = set(nunique[nunique > 1].index)
dual_mask = group_key.isin(bad_keys)
n_train = int((pool["_split"] == "train").sum())
n_val = int((pool["_split"] == "val").sum())
dropped_train = int((dual_mask & (pool["_split"] == "train")).sum())
dropped_val = int((dual_mask & (pool["_split"] == "val")).sum())
pool = pool.loc[~dual_mask].copy()
steps.append(_drop_row("groupes aux deux labels", "train", n_train, dropped_train))
steps.append(_drop_row("groupes aux deux labels", "val", n_val, dropped_val))
print(f"groupes aux deux labels retirés : {len(bad_keys)}")

empty_mask = pool["tweet"].map(lambda text: normalize_for_dedup(text) == "")
n_train = int((pool["_split"] == "train").sum())
n_val = int((pool["_split"] == "val").sum())
dropped_train = int((empty_mask & (pool["_split"] == "train")).sum())
dropped_val = int((empty_mask & (pool["_split"] == "val")).sum())
pool = pool.loc[~empty_mask].copy()
steps.append(_drop_row("textes vides", "train", n_train, dropped_train))
steps.append(_drop_row("textes vides", "val", n_val, dropped_val))

test_exact = set(test["tweet"].astype(str))
test_norm = set(test["tweet"].map(normalize_for_dedup))
leak_mask = pool["tweet"].astype(str).isin(test_exact) | pool["tweet"].map(normalize_for_dedup).isin(
    test_norm
)
n_train = int((pool["_split"] == "train").sum())
n_val = int((pool["_split"] == "val").sum())
dropped_train = int((leak_mask & (pool["_split"] == "train")).sum())
dropped_val = int((leak_mask & (pool["_split"] == "val")).sum())
pool = pool.loc[~leak_mask].copy()
steps.append(_drop_row("fuites test", "train", n_train, dropped_train))
steps.append(_drop_row("fuites test", "val", n_val, dropped_val))

train = pool.loc[pool["_split"] == "train"].drop(columns="_split").reset_index(drop=True)
val = pool.loc[pool["_split"] == "val"].drop(columns="_split").reset_index(drop=True)
display(pd.DataFrame(steps))

val_share = len(val) / (len(train) + len(val))
print(f"part de val dans train+val : {val_share:.4f}")
resplit = abs(val_share - VAL_SHARE) > VAL_TOLERANCE
if resplit:
    pool = pd.concat([train, val], ignore_index=True)
    train, val = train_test_split(
        pool,
        test_size=VAL_SHARE,
        random_state=SEED,
        stratify=pool["sarcasm"],
    )
    train = train.reset_index(drop=True)
    val = val.reset_index(drop=True)
    print("écart supérieur à un point : re-découpage 90/10 stratifié, graine 42, test non touché")
else:
    print("la part de val reste à environ 10 % : pas de re-découpage")

same_test = (
    list(test["id"]) == list(test_before["id"])
    and list(test["tweet"]) == list(test_before["tweet"])
    and list(test["sarcasm"]) == list(test_before["sarcasm"])
    and list(test["sentiment"]) == list(test_before["sentiment"])
    and list(test["dialect"]) == list(test_before["dialect"])
)
print(f"test identique à l'original : {same_test}")
print(f"ids test inchangés : {set(test['id']) == ids_before['test']}")
if resplit:
    print("train et val ont été redécoupés : les ids ne restent pas dans leur split d'origine")
else:
    print(f"train restant inclus dans le train initial : {set(train['id']) <= ids_before['train']}")
    print(f"val restante incluse dans la val initiale : {set(val['id']) <= ids_before['val']}")
    print(f"aucun id n'a changé de split : {set(train['id']).isdisjoint(ids_before['val'])}")

print(f"train / val / test : {len(train)} / {len(val)} / {len(test)}")
print(f"part train dans train+val : {len(train) / (len(train) + len(val)):.4f}")
print(f"part val dans train+val   : {len(val) / (len(train) + len(val)):.4f}")

overview_after = split_overview(train, val, test, "après")
display(pd.concat([overview_before, overview_after], ignore_index=True))

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
splits_dir = ROOT / "data" / "splits"
splits_dir.mkdir(parents=True, exist_ok=True)
train.to_csv(splits_dir / "train.csv", index=False)
val.to_csv(splits_dir / "val.csv", index=False)
train = pd.read_csv(splits_dir / "train.csv")
val = pd.read_csv(splits_dir / "val.csv")
for frame in (train, val):
    frame["tweet"] = frame["tweet"].fillna("").astype(str)
    frame["sarcasm"] = (
        frame["sarcasm"].astype(str).str.strip().str.upper().isin(["TRUE", "1", "T", "YES"])
    )
    frame["sentiment"] = frame["sentiment"].astype(str)
    frame["dialect"] = frame["dialect"].astype(str)

display(sarcasm_rate_table({"train": train, "val": val}))
display(sarcasm_rate_table({"train": train, "val": val}, by="dialect"))


In [ ]:
def strip_special(text):
    cleaned = strip_csv_quotes(text)
    cleaned = URL_RE.sub(" ", cleaned)
    cleaned = MENTION_RE.sub(" ", cleaned)
    cleaned = re.sub(r"(?<!\w)#(\w+)", r"\1", cleaned)
    return WS_RE.sub(" ", cleaned).strip()


preview = train[["tweet"]].copy()
preview["text"] = preview["tweet"].map(strip_special)
display(preview.loc[preview["tweet"] != preview["text"]].head(6))

for frame in (train, val):
    frame["text"] = frame["tweet"].map(strip_special)

In [ ]:
from collections import Counter

EMOJI_AR = {
    '2': '٢',
    'a': '',
    'A': 'أ',
    'Africa': 'أفريقيا',
    'alarm': 'منبه',
    'Americas': 'الأمريكتان',
    'and': 'و',
    'anger': 'غضب',
    'angry': 'غاضب',
    'anguished': 'متألم',
    'anxious': 'قلق',
    'apple': 'تفاحة',
    'Arab': 'عربي',
    'Arabia': 'العربية',
    'arrow': 'سهم',
    'arrows': 'أسهم',
    'articulated': 'مفصلي',
    'astonished': 'مندهش',
    'automobile': 'سيارة',
    'baby': 'طفل',
    'back': 'خلف',
    'backhand': 'ظهر اليد',
    'bag': 'حقيبة',
    'bags': 'حقائب',
    'ball': 'كرة',
    'balloon': 'بالون',
    'bandage': 'ضمادة',
    'bank': 'بنك',
    'beaming': 'مشرق',
    'beating': 'خافق',
    'bell': 'جرس',
    'bellhop': 'حامل حقائب',
    'beverage': 'مشروب',
    'biceps': 'عضلة',
    'big': 'كبير',
    'birthday': 'عيد ميلاد',
    'black': 'أسود',
    'blond': 'أشقر',
    'blood': 'دم',
    'blossom': 'زهرة',
    'blowing': 'ينفخ',
    'blue': 'أزرق',
    'board': 'لوحة',
    'bomb': 'قنبلة',
    'book': 'كتاب',
    'books': 'كتب',
    'bottle': 'زجاجة',
    'bouquet': 'باقة',
    'bowing': 'ينحني',
    'boy': 'ولد',
    'broken': 'مكسور',
    'but': 'لكن',
    'butterfly': 'فراشة',
    'button': 'زر',
    'cake': 'كعكة',
    'camera': 'كاميرا',
    'car': 'سيارة',
    'cat': 'قط',
    'check': 'صح',
    'cherry': 'كرز',
    'chick': 'كتكوت',
    'Christmas': 'عيد الميلاد',
    'circle': 'دائرة',
    'circled': 'محاط',
    'clapper': 'كلاكيت',
    'clapping': 'تصفيق',
    'Claus': 'كلوز',
    'clock': 'ساعة',
    'clockwise': 'مع عقارب الساعة',
    'closed': 'مغلق',
    'cloud': 'غيمة',
    'clover': 'برسيم',
    'clown': 'مهرج',
    'collision': 'اصطدام',
    'confetti': 'قصاصات',
    'confounded': 'مرتبك',
    'confused': 'حائر',
    'cork': 'سدادة',
    'couple': 'زوج',
    'crescent': 'هلال',
    'cross': 'صليب',
    'crossbones': 'عظام متقاطعة',
    'crown': 'تاج',
    'crying': 'يبكي',
    'crystal': 'كريستال',
    'curly': 'مجعد',
    'cyclone': 'إعصار',
    'dancing': 'يرقص',
    'dark': 'داكن',
    'decoration': 'زخرفة',
    'diamond': 'معين',
    'disappointed': 'خائب',
    'dizzy': 'دائر',
    'dog': 'كلب',
    'double': 'مزدوج',
    'dove': 'حمامة',
    'down': 'أسفل',
    'downcast': 'مغموم',
    'droplets': 'قطرات',
    'drops': 'قطرات',
    'eggplant': 'باذنجان',
    'Egypt': 'مصر',
    'eight': 'ثمانية',
    'eighteen': 'ثمانية عشر',
    'Emirates': 'الإمارات',
    'entry': 'دخول',
    'envelope': 'ظرف',
    'Europe': 'أوروبا',
    'evil': 'شرير',
    'ewe': 'نعجة',
    'exclamation': 'تعجب',
    'exploding': 'ينفجر',
    'expressionless': 'بلا تعبير',
    'eyes': 'عيون',
    'face': 'وجه',
    'facing': 'مقابل',
    'fallen': 'ساقط',
    'fear': 'خوف',
    'fearful': 'خائف',
    'finger': 'إصبع',
    'fire': 'نار',
    'fist': 'قبضة',
    'flag': 'علم',
    'flash': 'وميض',
    'flexed': 'مثنية',
    'floor': 'أرض',
    'flushed': 'محمر',
    'fluttering': 'يرفرف',
    'folded': 'متشابكة',
    'food': 'طعام',
    'four': 'أربعة',
    'frog': 'ضفدع',
    'from': 'من',
    'front': 'أمام',
    'frowning': 'عابس',
    'full': 'بدر',
    'gem': 'جوهرة',
    'gesturing': 'يومئ',
    'gift': 'هدية',
    'girl': 'بنت',
    'glasses': 'نظارات',
    'globe': 'كرة أرضية',
    'glowing': 'متوهج',
    'green': 'أخضر',
    'grimacing': 'متجهم',
    'grinning': 'مبتسم',
    'growing': 'ينمو',
    'hair': 'شعر',
    'hand': 'يد',
    'hands': 'يدان',
    'hat': 'قبعة',
    'head': 'رأس',
    'headphone': 'سماعة',
    'hear': 'يسمع',
    'heart': 'قلب',
    'hearts': 'قلوب',
    'herb': 'عشبة',
    'hibiscus': 'كركديه',
    'high': 'عال',
    'hole': 'حفرة',
    'horns': 'قرون',
    'horse': 'حصان',
    'hot': 'حار',
    'hugging': 'يعانق',
    'hundred': 'مئة',
    'hushed': 'صامت',
    'in': 'في',
    'incoming': 'قادم',
    'index': 'سبابة',
    'jack': 'جاك',
    'joy': 'فرح',
    'key': 'مفتاح',
    'keyboard': 'لوحة مفاتيح',
    'keycap': 'مفتاح رقم',
    'kiss': 'قبلة',
    'kissing': 'يقبل',
    'Kuwait': 'الكويت',
    'label': 'ملصق',
    'lantern': 'فانوس',
    'laptop': 'حاسوب',
    'large': 'كبير',
    'Latvia': 'لاتفيا',
    'laughing': 'يضحك',
    'leaf': 'ورقة',
    'Lebanon': 'لبنان',
    'left': 'يسار',
    'letter': 'رسالة',
    'Libya': 'ليبيا',
    'light': 'فاتح',
    'lightning': 'برق',
    'locked': 'مقفل',
    'loop': 'حلقة',
    'lorry': 'شاحنة',
    'loudly': 'بصوت عال',
    'love': 'حب',
    'M': 'م',
    'man': 'رجل',
    'mark': 'علامة',
    'mask': 'كمامة',
    'medal': 'ميدالية',
    'medical': 'طبي',
    'medium': 'متوسط',
    'megaphone': 'مكبر صوت',
    'meridians': 'خطوط طول',
    'Mexico': 'المكسيك',
    'microphone': 'ميكروفون',
    'middle': 'وسطى',
    'military': 'عسكري',
    'moai': 'مواي',
    'mobile': 'جوال',
    'money': 'مال',
    'monkey': 'قرد',
    'moon': 'قمر',
    'mouth': 'فم',
    'movie': 'فيلم',
    'musical': 'موسيقي',
    'nail': 'ظفر',
    'national': 'وطني',
    'nauseated': 'متقزز',
    'nerd': 'مثقف',
    'neutral': 'محايد',
    'new': 'جديد',
    'newspaper': 'جريدة',
    'nib': 'ريشة',
    'nine': 'تسعة',
    'no': 'لا',
    'nose': 'أنف',
    'note': 'ملاحظة',
    'notebook': 'دفتر',
    'notepad': 'مفكرة',
    'notes': 'نوتات',
    'o': 'و',
    "o'clock": 'تمام',
    'of': 'من',
    'ogre': 'غول',
    'OK': 'موافق',
    'on': 'على',
    'oncoming': 'مقبل',
    'one': 'واحد',
    'open': 'مفتوح',
    'orange': 'برتقالي',
    'over': 'فوق',
    'package': 'طرد',
    'Palestinian': 'فلسطيني',
    'palm': 'كف',
    'palms': 'كفان',
    'panda': 'باندا',
    'park': 'حديقة',
    'party': 'حفلة',
    'partying': 'يحتفل',
    'pensive': 'متأمل',
    'persevering': 'مثابر',
    'person': 'شخص',
    'phone': 'هاتف',
    'pile': 'كومة',
    'pinching': 'يقرص',
    'pistol': 'مسدس',
    'pleading': 'متوسل',
    'pointed': 'مسنن',
    'pointing': 'يشير',
    'points': 'نقاط',
    'police': 'شرطة',
    'polish': 'طلاء',
    'poo': 'براز',
    'popper': 'مفرقعات',
    'popping': 'تنفتح',
    'pouting': 'عابس',
    'princess': 'أميرة',
    'prohibited': 'ممنوع',
    'purple': 'بنفسجي',
    'pushpin': 'دبوس',
    'question': 'سؤال',
    'radio': 'راديو',
    'rain': 'مطر',
    'raised': 'مرفوع',
    'raising': 'يرفع',
    'recycling': 'إعادة تدوير',
    'red': 'أحمر',
    'relieved': 'مرتاح',
    'revolving': 'دوار',
    'ribbon': 'شريط',
    'rice': 'أرز',
    'right': 'يمين',
    'ring': 'خاتم',
    'robot': 'روبوت',
    'rocket': 'صاروخ',
    'rolled': 'ملفوف',
    'rolling': 'يدور',
    'rose': 'وردة',
    'round': 'مستدير',
    'running': 'يركض',
    'sad': 'حزين',
    'Santa': 'بابا نويل',
    'Saudi': 'السعودية',
    'savoring': 'يتذوق',
    'score': 'نتيجة',
    'screaming': 'يصرخ',
    'see': 'يرى',
    'seedling': 'شتلة',
    'shamrock': 'برسيم',
    'sheaf': 'حزمة',
    'shopping': 'تسوق',
    'showing': 'يظهر',
    'shushing': 'يسكت',
    'sign': 'لافتة',
    'skin': 'بشرة',
    'skull': 'جمجمة',
    'sleeping': 'نائم',
    'sleepy': 'نعسان',
    'slightly': 'قليلا',
    'small': 'صغير',
    'smiling': 'مبتسم',
    'smirking': 'بابتسامة ماكرة',
    'snow': 'ثلج',
    'snowflake': 'ندفة ثلج',
    'snowman': 'رجل ثلج',
    'soccer': 'كرة قدم',
    'Spain': 'إسبانيا',
    'sparkle': 'بريق',
    'sparkles': 'بريق',
    'sparkling': 'لامع',
    'speak': 'يتكلم',
    'spiral': 'لولب',
    'square': 'مربع',
    'squinting': 'يضيق عينيه',
    'star': 'نجمة',
    'States': 'الولايات',
    'steam': 'بخار',
    'stone': 'حجر',
    'suit': 'مجموعة',
    'sun': 'شمس',
    'sunflower': 'دوار الشمس',
    'sunglasses': 'نظارات شمسية',
    'sweat': 'عرق',
    'symbol': 'رمز',
    'tear': 'دمعة',
    'tears': 'دموع',
    'technologist': 'تقني',
    'telephone': 'هاتف',
    'television': 'تلفاز',
    'tennis': 'تنس',
    'Territories': 'الأراضي',
    'the': '',
    'thinking': 'يفكر',
    'thirty': 'ثلاثون',
    'thought': 'فكرة',
    'thumbs': 'إبهام',
    'tired': 'متعب',
    'together': 'معا',
    'tone': 'لون',
    'tongue': 'لسان',
    'top': 'علوي',
    'trade': 'تجارة',
    'tree': 'شجرة',
    'triangle': 'مثلث',
    'triangular': 'مثلثي',
    'tulip': 'توليب',
    'turban': 'عمامة',
    'Turkey': 'تركيا',
    'twelve': 'اثنا عشر',
    'two': 'اثنان',
    'type': 'فصيلة',
    'umbrella': 'مظلة',
    'unamused': 'غير مستمتع',
    'under': 'تحت',
    'United': 'المتحدة',
    'unlocked': 'مفتوح',
    'up': 'أعلى',
    'upside': 'مقلوب',
    'vertical': 'عمودي',
    'victory': 'نصر',
    'video': 'فيديو',
    'voltage': 'جهد',
    'walking': 'يمشي',
    'water': 'ماء',
    'waving': 'يلوح',
    'wearing': 'يرتدي',
    'weary': 'منهك',
    'white': 'أبيض',
    'wind': 'ريح',
    'winking': 'يغمز',
    'with': 'مع',
    'without': 'بدون',
    'woman': 'امرأة',
    'worried': 'قلق',
    'wrapped': 'مغلف',
    'writing': 'كتابة',
    'yellow': 'أصفر',
    'zany': 'مجنون',
    'zzz': 'نوم',
    'Bahrain': 'البحرين',
    'Italy': 'إيطاليا',
    'Oman': 'عمان',
    'Qatar': 'قطر',
    'beer': 'جعة',
    'behind': 'خلف',
    'bird': 'طائر',
    'boat': 'قارب',
    'brown': 'بني',
    'bulb': 'مصباح',
    'cane': 'عكاز',
    'cherries': 'كرز',
    'clinking': 'قرع',
    'cold': 'بارد',
    'counterclockwise': 'عكس عقارب الساعة',
    'crossed': 'متقاطع',
    'deaf': 'أصم',
    'disguised': 'متنكر',
    'downwards': 'للأسفل',
    'drooling': 'يسيل لعابه',
    'drop': 'قطرة',
    'droplet': 'قطرة',
    'duck': 'بطة',
    'facepalming': 'كف على الوجه',
    'female': 'أنثى',
    'fingers': 'أصابع',
    'flower': 'زهرة',
    'getting': 'يصير',
    'goblin': 'عفريت',
    'grapes': 'عنب',
    'halo': 'هالة',
    'handshake': 'مصافحة',
    'knocked': 'مصدوم',
    'link': 'رابط',
    'mailbox': 'صندوق بريد',
    'male': 'ذكر',
    'mango': 'مانجو',
    'maple': 'قيقب',
    'massage': 'تدليك',
    'melon': 'شمام',
    'mugs': 'كؤوس',
    'otter': 'قضاعة',
    'out': 'خارج',
    'pear': 'كمثرى',
    'rainbow': 'قوس قزح',
    'registered': 'مسجل',
    'repeat': 'تكرار',
    'rowing': 'تجديف',
    'shrugging': 'يهز كتفيه',
    'snake': 'ثعبان',
    'speech': 'كلام',
    'standing': 'واقف',
    'strawberry': 'فراولة',
    'struck': 'مصعوق',
    'syringe': 'حقنة',
    'violin': 'كمان',
    'warning': 'تحذير',
    'watch': 'ساعة يد',
    'wilted': 'ذابلة',
}


def emoji_symbols(text):
    return [hit["emoji"] for hit in emoji.emoji_lis(str(text).replace("\ufe0f", ""))]


def arabic_emoji_name(symbol):
    english = emoji.demojize(symbol).strip(":")
    parts = english.replace("-", " ").replace("_", " ").replace("’", "'").split()
    words = []
    for part in parts:
        token = part.strip("()")
        words.append(EMOJI_AR.get(token, EMOJI_AR.get(token.lower(), token)))
    return " ".join(word for word in words if word).strip()


found = Counter()
for text in train["text"]:
    found.update(emoji_symbols(text))

arabic_by_emoji = {symbol: arabic_emoji_name(symbol) for symbol in found}
print(f"emojis distincts, un nom chacun : {len(arabic_by_emoji)}")

emoji_map = pd.DataFrame(
    {
        "emoji": list(found.keys()),
        "arabe": [arabic_by_emoji[symbol] for symbol in found.keys()],
        "n_train": list(found.values()),
    }
).sort_values("n_train", ascending=False)
display(emoji_map.head(20))

top = emoji_map.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top["arabe"], top["n_train"])
ax.set_xlabel("occurrences dans le train")
ax.set_title("Emojis les plus fréquents")
fig.tight_layout()
display(fig)
plt.close(fig)


def map_emojis(text):
    value = str(text).replace("\ufe0f", "")
    for symbol in set(emoji_symbols(value)):
        name = arabic_by_emoji.get(symbol)
        if name:
            value = value.replace(symbol, " " + name + " ")
    return WS_RE.sub(" ", value).strip()


preview = train.loc[train["text"].map(lambda text: len(emoji_symbols(text)) > 0), ["text"]].head(6).copy()
preview["apres"] = preview["text"].map(map_emojis)
display(preview)

for frame in (train, val):
    frame["text"] = frame["text"].map(map_emojis)


In [ ]:
def strip_arabic_noise(text):
    cleaned = DIACRITIC_RE.sub("", str(text))
    cleaned = cleaned.replace(TATWEEL, "")
    cleaned = re.sub(r"([^\W\d_])\1{2,}", r"\1", cleaned)
    return WS_RE.sub(" ", cleaned).strip()


noise = train["text"].map(
    lambda text: DIACRITIC_RE.search(text) is not None
    or TATWEEL in text
    or REPEATED_LETTER_RE.search(text) is not None
)
preview = train.loc[noise, ["text"]].head(6).copy()
preview["apres"] = preview["text"].map(strip_arabic_noise)
display(preview)

for frame in (train, val):
    frame["text"] = frame["text"].map(strip_arabic_noise)

print(f"textes vides après nettoyage — train : {int((train['text'] == '').sum())}")
print(f"textes vides après nettoyage — val   : {int((val['text'] == '').sum())}")

### Ponctuation

Une fois les lignes retirées, on enlève la ponctuation de la colonne `text` de train et de val. Le test n'est pas concerné. `\p{P}` couvre `!`, `?`, `-`, `_` et la ponctuation Unicode (virgule arabe, point d'interrogation arabe, guillemets). Les lettres arabes et les chiffres restent.

In [ ]:
try:
    import regex as _regex

    _PUNCT_RE = _regex.compile(r"[\p{P}_]+")

    def strip_punctuation(text):
        return WS_RE.sub(" ", _PUNCT_RE.sub(" ", str(text))).strip()

except ImportError:
    import unicodedata

    def strip_punctuation(text):
        kept = []
        for char in str(text):
            if char == "_" or unicodedata.category(char).startswith("P"):
                kept.append(" ")
            else:
                kept.append(char)
        return WS_RE.sub(" ", "".join(kept)).strip()


preview = train[["text"]].copy()
preview["apres"] = preview["text"].map(strip_punctuation)
display(preview.loc[preview["text"] != preview["apres"]].head(6))

for frame in (train, val):
    frame["text"] = frame["text"].map(strip_punctuation)

print(f"textes vides après ponctuation — train : {int((train['text'] == '').sum())}")
print(f"textes vides après ponctuation — val   : {int((val['text'] == '').sum())}")
print(f"test sans colonne text : {'text' not in test.columns} — n={len(test)}")


In [ ]:
LATIN_LETTER_AR = {
    "a": "ا", "b": "ب", "c": "ك", "d": "د", "e": "ي",
    "f": "ف", "g": "ج", "h": "ه", "i": "ي", "j": "ج",
    "k": "ك", "l": "ل", "m": "م", "n": "ن", "o": "و",
    "p": "ب", "q": "ق", "r": "ر", "s": "س", "t": "ت",
    "u": "و", "v": "ف", "w": "و", "x": "كس", "y": "ي", "z": "ز",
}
DIGIT_AR = str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩")
letter_table = pd.DataFrame({"latin": list(LATIN_LETTER_AR), "arabe": list(LATIN_LETTER_AR.values())})
digit_table = pd.DataFrame({"latin": list("0123456789"), "arabe": list("٠١٢٣٤٥٦٧٨٩")})
display(letter_table)
display(digit_table)


def latin_token_to_arabic(token):
    pieces = []
    for char in token:
        if char.isdigit():
            pieces.append(char.translate(DIGIT_AR))
        else:
            pieces.append(LATIN_LETTER_AR.get(char.lower(), char))
    return "".join(pieces)


def map_latin(text):
    return re.sub(r"[A-Za-z0-9]+", lambda match: latin_token_to_arabic(match.group(0)), str(text))


has_latin = train["text"].map(lambda text: re.search(r"[A-Za-z0-9]", text) is not None)
preview = train.loc[has_latin, ["text"]].head(6).copy()
preview["apres"] = preview["text"].map(map_latin)
display(preview)

for frame in (train, val):
    frame["text"] = frame["text"].map(map_latin)


## TF-IDF

Chaque tweet devient un vecteur creux de poids TF-IDF. Deux vectoriseurs sont ajustés sur `train` seulement, puis collés côte à côte.

**Mots, `ngram_range=(1, 2)`.** `analyzer="word"` découpe le texte en mots. `(1, 2)` garde les unigrammes (1 mot) et les bigrammes (2 mots consécutifs). Sur « مش عارف », les traits sont `مش`, `عارف` et `مش عارف`.

**Caractères, `ngram_range=(2, 5)`.** `analyzer="char"` ne découpe pas en mots. Une fenêtre glisse sur la chaîne et retient chaque sous-chaîne de longueur 2, 3, 4 et 5. `(2, 5)` veut donc dire : n-grammes de caractères du bigramme au 5-gramme. Sur `هههه` on obtient par exemple `هه`, `ههه` et `هههه`. Ces traits attrapent ce que les mots ratent : rires (`ههه`), lettres allongées, préfixes et suffixes dialectaux, émojis, racines écrites autrement.

**`FeatureUnion`.** Elle concatène les deux matrices, colonne par colonne. Un tweet devient `[TF-IDF mots | TF-IDF caractères]`. La régression logistique voit les deux vocabulaires d'un coup. `fit_transform` apprend les deux vocabulaires sur `train["text"]` uniquement.


In [ ]:
word_vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1, 2))
char_vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 5))
tfidf = FeatureUnion([("word", word_vectorizer), ("char", char_vectorizer)])
X_train = tfidf.fit_transform(train["text"])

print(f"documents train : {X_train.shape[0]}")
print(f"traits          : {X_train.shape[1]}")
print(f"vocabulaire mots        : {len(word_vectorizer.vocabulary_)}")
print(f"vocabulaire caractères  : {len(char_vectorizer.vocabulary_)}")

## Métadonnées catégorielles

Le dialecte (`dialect`) et l'émotion (`sentiment` : `NEG`, `NEU`, `POS`) ne sont pas du texte. Ce sont des catégories. La régression logistique ne lit que des nombres, donc chaque modalité devient une colonne binaire.

**`OneHotEncoder(drop="first")`.** Pour une variable à k modalités, l'encodeur produit k − 1 colonnes. `drop="first"` retire la première modalité, dans l'ordre alphabétique des catégories vues sur `train`. Cette modalité sert de référence : elle est codée par des zéros sur les autres colonnes de la même variable. On évite ainsi une colonne redondante, puisque la somme des indicatrices vaudrait toujours 1. Sur ce jeu, les colonnes gardées ont la forme `dialect_gulf`, `dialect_levant`, `sentiment_NEU`, `sentiment_POS`. `handle_unknown="ignore"` : une modalité absente du train, si elle apparaît en validation, est codée par des zéros.

**`ColumnTransformer`.** La `FeatureUnion` déjà construite ne voit que `text` : elle colle le TF-IDF des mots au TF-IDF des caractères. Le `ColumnTransformer` envoie `text` à cette union, et `dialect` plus `sentiment` à l'encodeur, puis concatène les deux blocs horizontalement. Un tweet devient `[TF-IDF mots | TF-IDF caractères | indicatrices]`. `fit_transform` n'apprend que sur `train`. `sparse_threshold=1` garde une matrice creuse : le TF-IDF est presque entièrement nul, et `liblinear` la lit telle quelle.


In [ ]:
one_hot = OneHotEncoder(drop="first", handle_unknown="ignore")
features = ColumnTransformer(
    [
        ("tfidf", tfidf, "text"),
        ("meta", one_hot, ["dialect", "sentiment"]),
    ],
    # 1 : ne pas densifier. Le TF-IDF est creux ; liblinear lit une matrice creuse.
    sparse_threshold=1.0,
)
X_train = features.fit_transform(train)

meta_names = list(
    features.named_transformers_["meta"].get_feature_names_out(["dialect", "sentiment"])
)
print("indicatrices :", meta_names)
print(f"documents train : {X_train.shape[0]}")
print(f"traits TF-IDF   : {X_train.shape[1] - len(meta_names)}")
print(f"traits one-hot  : {len(meta_names)}")
print(f"traits au total : {X_train.shape[1]}")


## Entraînement

### Baseline triviale

Un classifieur **trivial** ne lit pas le tweet. Il rend toujours la même réponse. Ici, la réponse est toujours « non sarcastique ». Ce n'est pas un modèle à déployer : c'est le plancher. L'accuracy et le F1 de la régression logistique se lisent par rapport à ce plancher.

`trivial_pred = np.zeros(len(val), dtype=bool)` construit un tableau de la longueur de `val`, rempli de zéros et stocké en booléens. En booléen, `0` vaut `False`. Chaque tweet de validation est donc prédit non sarcastique.

`trivial_proba = np.zeros(len(val), dtype=float)` est le même choix, exprimé en probabilités : `0.0` sur chaque ligne. Le baseline affirme que la probabilité de sarcasme est nulle partout. On le garde pour le comparer aux probabilités de la régression logistique, qui varient entre 0 et 1.

### Pourquoi `astype(bool)`

Au chargement, `isin(["TRUE", "1", "T", "YES"])` a déjà fait de `sarcasm` une série de `True` / `False`. `.astype(bool)` fige ce type, et `.to_numpy()` quitte pandas pour un tableau NumPy. `trivial_pred` est un tableau de booléens : `y_val` doit avoir le même type, afin qu'`accuracy_score` et `f1_score` comparent l'étiquette vraie et l'étiquette prédite. `pos_label=True` désigne ensuite cette valeur `True` comme la classe sarcastique.

### Paramètres de `f1_score`

`f1_score(y_val, trivial_pred, pos_label=True, zero_division=0)`

Le F1 est la moyenne harmonique de la précision et du rappel d'**une** classe, la classe positive.

- `y_val` : étiquettes vraies.
- `trivial_pred` : étiquettes prédites. Ici, uniquement des `False`.
- `pos_label=True` : la classe dont on mesure le F1. `True` veut dire sarcastique. Le score ignore la classe « non sarcastique ». Aucune prédiction n'est `True`, le rappel du sarcasme vaut 0, et le F1 sarcastique vaut 0. L'accuracy peut rester haute, parce qu'elle compte aussi les non-sarcastiques bien classés.
- `zero_division=0` : la précision divise les vrais positifs par le nombre de prédictions positives. Ce dénominateur vaut 0 ici, puisqu'aucun tweet n'est prédit sarcastique. L'argument fixe alors le résultat à 0.


In [ ]:
y_val = val["sarcasm"].astype(bool).to_numpy()
trivial_pred = np.zeros(len(val), dtype=bool)
trivial_proba = np.zeros(len(val), dtype=float)
trivial_acc = accuracy_score(y_val, trivial_pred)
trivial_f1 = f1_score(y_val, trivial_pred, pos_label=True, zero_division=0)
print(f"accuracy val : {trivial_acc:.4f}")
print(f"F1 sarcastique val : {trivial_f1:.4f}")

In [ ]:
# transform applique, sur val, les colonnes déjà apprises sur train :
# le vocabulaire TF-IDF et les indicatrices de dialecte et d'émotion.
X_val = features.transform(val)
clf = LogisticRegression(
    # "balanced" : poids inversement proportionnel à la fréquence de la classe.
    # Le sarcasme est minoritaire ; une erreur sur un sarcastique pèse davantage.
    class_weight="balanced",
    # liblinear : algorithme qui minimise la perte, adapté au cas binaire
    # et à une matrice creuse (beaucoup de zéros, ce qu'est le TF-IDF).
    solver="liblinear",
    max_iter=1000,
    random_state=SEED,
)
started = time.perf_counter()
clf.fit(X_train, train["sarcasm"].astype(bool))
elapsed = time.perf_counter() - started
# coef_ : un poids par colonne de la matrice concaténée (TF-IDF et indicatrices).
# intercept_ : le biais, un seul nombre.
# n_params compte ces deux blocs : c'est le nombre de valeurs réglées par fit.
n_params = int(clf.coef_.size + clf.intercept_.size)

# classes_ suit l'ordre appris, en général [False, True].
# sarcasm_index désigne la colonne de predict_proba qui correspond à True.
classes = list(clf.classes_)
sarcasm_index = classes.index(True)
# val_proba[i] = P(sarcastique | tweet i), un nombre entre 0 et 1.
val_proba = clf.predict_proba(X_val)[:, sarcasm_index]

# Le seuil est la barre de décision. La proba seule ne dit pas oui ou non :
# on prédit « sarcastique » quand elle est >= seuil.
# Grille 0.01, 0.02, ..., 0.99. On garde le seuil au meilleur F1 sarcastique sur val.
grid = np.round(np.arange(0.01, 1.00, 0.01), 2)
best_threshold = float(grid[0])
best_f1 = -1.0
for threshold in grid:
    # val_proba >= threshold : un booléen par tweet, à ce seuil-là.
    f1 = f1_score(y_val, val_proba >= threshold, pos_label=True, zero_division=0)
    if f1 > best_f1:
        best_f1 = float(f1)
        best_threshold = float(threshold)
# Prédiction retenue avec la barre qui a gagné.
val_pred = val_proba >= best_threshold

print(f"seuil : {best_threshold:.2f}")
print(f"F1 sarcastique val : {best_f1:.4f}")
print(f"temps d'entraînement : {elapsed:.1f} s")
print(f"paramètres entraînés : {n_params}")

In [ ]:
cm = confusion_matrix(y_val, val_pred, labels=[False, True])
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    cm,
    display_labels=["non sarcastique", "sarcastique"],
).plot(ax=ax, colorbar=False)
ax.set_title("Matrice de confusion — validation")
fig.tight_layout()
display(fig)
plt.close(fig)
